# 04. Rezultatele finale și legătura cu HNP

Ultimul notebook adună rezultatele din toate etapele și trece de la prefixele estimate de CNN la instanța HNP. Separăm validarea ecuațiilor, disponibilitatea reducerii LLL și recuperarea efectivă a cheii.

## Regula de raportare

Etapa HNP are trei niveluri de verificare:

1. **Formularea HNP.** Verificăm dacă cheia din oracle satisface exact relațiile și vectorul țintă al lattice-ului.
2. **Reducerea LLL.** Verificăm dacă biblioteca necesară este disponibilă și dacă baza poate fi redusă.
3. **Recuperarea cheii.** Acceptăm un rezultat doar când candidatul returnat de solver trece toate constrângerile HNP.

O instanță HNP validă nu dovedește singură că reducerea a recuperat cheia.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "informatica").is_dir() and (p / "notebooks").is_dir()
)

ECDSA_DIR = PROJECT_ROOT / "informatica" / "ECDSA"
CNN_DIR = PROJECT_ROOT / "informatica" / "cnn"
ARTIFACTS_DIR = PROJECT_ROOT / "informatica" / "artifacts"
RESULTS_DIR = PROJECT_ROOT / "results"

for path in (ECDSA_DIR, CNN_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import contextlib
import io
import json

import pandas as pd
import numpy as np

from dataset_generator3 import CURVE, load_public_dataset, load_ground_truth
from hnp_utils import compute_hnp_terms
from Lattice_key6 import FPYLLL_AVAILABLE, HNPConfig, HNPLatticeSolver
import theory_to_experiment_bridge8 as bridge
import diophantine_experiments7 as diophantine

public_records = load_public_dataset()
oracle = load_ground_truth()

baseline_results = json.loads((ARTIFACTS_DIR / "baseline_results.json").read_text(encoding="utf-8"))
cnn_results = json.loads((ARTIFACTS_DIR / "cnn_notebook_results.json").read_text(encoding="utf-8"))

## Rezultatele CNN

Păstrăm atât acuratețea pe bit, cât și acuratețea exactă a prefixului de 12 biți. Un decoder binar aleator are 50% acuratețe pe bit, iar șansa ca toate cele 12 predicții să fie corecte independent este $2^{-12}$, adică aproximativ 0,0244%.

In [3]:
cnn_metrics = cnn_results["metrics"]

cnn_summary = pd.DataFrame([
    ["Train", cnn_metrics["train_bit_accuracy"], cnn_metrics["train_prefix_accuracy"], cnn_metrics["train_samples"]],
    ["Validation", cnn_metrics["validation_bit_accuracy"], cnn_metrics["validation_prefix_accuracy"], cnn_metrics["validation_samples"]],
    ["Test", cnn_metrics["test_bit_accuracy"], cnn_metrics["test_prefix_accuracy"], cnn_metrics["test_samples"]],
], columns=["set", "acuratețe_bit", "acuratețe_prefix", "eșantioane"])

display(cnn_summary.style.format({
    "acuratețe_bit": "{:.4f}",
    "acuratețe_prefix": "{:.4f}",
}))

,set,acuratețe_bit,acuratețe_prefix,eșantioane
0,Train,0.9658,0.6607,112
1,Validation,0.9583,0.5417,24
2,Test,0.9826,0.7917,24


## Construirea instanței HNP

Pentru fiecare semnătură calculăm

$$
t_i = s_i^{-1}r_i \pmod n, \qquad
u_i = s_i^{-1}z_i \pmod n, \qquad
a_i = \bar{k}_i\,2^{256-\ell}.
$$

Atunci cheia privată satisface

$$
t_i d + u_i - a_i \equiv \delta_i \pmod n, \qquad
0 \le \delta_i < 2^{256-\ell}.
$$

Implementarea proiectului folosește un lattice de dimensiune $m+2$, unde $m$ este numărul de semnături selectate. Recuperarea este formulată ca o problemă de vector scurt și redusă cu LLL.

In [4]:
selected = public_records[:40]
leaked_bits = int(selected[0]["leaked_bits"])
q = CURVE.n
hidden_bits = q.bit_length() - leaked_bits
prefixes_oracle = {
    int(k): int(v["leaked_nonce_prefix"])
    for k, v in oracle["samples"].items()
}

t_list = []
u_list = []
a_list = []

for record in selected:
    sample_id = int(record["sample_id"])
    t_i, u_i, a_i = compute_hnp_terms(
        int(record["r"]),
        int(record["s"]),
        int(record["z"]),
        prefixes_oracle[sample_id],
        q,
        leaked_bits,
    )
    t_list.append(t_i)
    u_list.append(u_i)
    a_list.append(a_i)

private_key = int(oracle["private_key"])
solver = HNPLatticeSolver(
    HNPConfig(leaked_bits=leaked_bits, num_samples=len(selected))
)

relation_ok = solver.verify_expected_vector(
    private_key,
    t_list,
    u_list,
    a_list,
)

hnp_instance_summary = pd.DataFrame([
    ["număr semnături", len(selected)],
    ["biți MSB disponibili", leaked_bits],
    ["biți necunoscuți", hidden_bits],
    ["B = 2^(256-ℓ)", solver.config.bound],
    ["M", solver.config.embedding],
    ["dimensiunea lattice-ului", len(t_list) + 2],
    ["ecuația HNP verificată", relation_ok],
], columns=["parametru", "valoare"])

display(hnp_instance_summary)
assert relation_ok

,parametru,valoare
0,număr semnături,40
1,biți MSB disponibili,12
2,biți necunoscuți,244
3,B = 2^(256-ℓ),2826955303645414927333276001188669625323974235...
4,M,3273390607896141870013189696827599152204417714...
5,dimensiunea lattice-ului,42
6,ecuația HNP verificată,True


## Verificarea reducerii LLL

Repository-ul declară `fpylll` ca dependență pentru reducerea lattice-ului. În mediul curent biblioteca nu este instalată. SymPy rămâne folosit pentru validările auxiliare și nu este tratat ca înlocuitor pentru reducerea de dimensiune mare.

Documentația [`fpylll`](https://fpylll.readthedocs.io/en/latest/) descrie interfața `IntegerMatrix` și `LLL.reduction`.

In [5]:
if FPYLLL_AVAILABLE:
    try:
        reduced = solver.reduce(
            solver.build_basis_matrix(t_list, u_list, a_list)
        )
        reduced_available = True
        reduction_message = "Reducerea LLL a fost executată."
    except Exception as exc:
        reduced_available = False
        reduction_message = f"Reducerea a eșuat: {exc}"
else:
    reduced_available = False
    reduction_message = "fpylll nu este instalat în mediul curent."

print(reduction_message)

fpylll nu este instalat în mediul curent.


## Testarea prefixelor generate de CNN

HNP trebuie să primească prefixele estimate de CNN. Aici folosim predicțiile salvate pentru setul de test și verificăm mai întâi dacă avem suficiente semnături pentru configurația HNP de 40 de eșantioane. Setul de test conține acum 40 de ID-uri, exact cât cere configurația HNP.

Dacă numărul de predicții este suficient, reducerea LLL depinde în continuare de disponibilitatea `fpylll`.

In [6]:
cnn_prefixes = {
    int(item["sample_id"]): int(item["predicted_prefix"])
    for item in cnn_results["predictions"]
}

cnn_selected = [
    record for record in public_records
    if int(record["sample_id"]) in cnn_prefixes
][:40]

cnn_hnp_ready = len(cnn_selected) >= 40

if cnn_hnp_ready:
    cnn_t = []
    cnn_u = []
    cnn_a = []

    for record in cnn_selected:
        sample_id = int(record["sample_id"])
        t_i, u_i, a_i = compute_hnp_terms(
            int(record["r"]),
            int(record["s"]),
            int(record["z"]),
            cnn_prefixes[sample_id],
            q,
            leaked_bits,
        )
        cnn_t.append(t_i)
        cnn_u.append(u_i)
        cnn_a.append(a_i)

    cnn_solver = HNPLatticeSolver(
        HNPConfig(leaked_bits=leaked_bits, num_samples=40)
    )

    cnn_recovery = None
    cnn_reduction_error = None

    if FPYLLL_AVAILABLE:
        try:
            cnn_recovery = cnn_solver.solve(cnn_t, cnn_u, cnn_a)
        except Exception as exc:
            cnn_reduction_error = str(exc)
    else:
        cnn_reduction_error = "fpylll nu este instalat."

    cnn_candidate_valid = (
        cnn_recovery is not None
        and cnn_solver.validate_candidate(cnn_recovery, cnn_t, cnn_u, cnn_a)
    )
else:
    cnn_recovery = None
    cnn_candidate_valid = False
    cnn_reduction_error = "Mai puțin de 40 de predicții CNN disponibile."

hnp_cnn_summary = pd.DataFrame([
    ["predicții CNN disponibile", len(cnn_prefixes)],
    ["eșantioane folosite în HNP", min(40, len(cnn_selected))],
    ["reducere LLL disponibilă", FPYLLL_AVAILABLE],
    ["candidat returnat", cnn_recovery is not None],
    ["candidat validat", cnn_candidate_valid],
    ["mesaj reducere", cnn_reduction_error or "fără eroare"],
], columns=["verificare", "rezultat"])

display(hnp_cnn_summary)

,verificare,rezultat
0,predicții CNN disponibile,24
1,eșantioane folosite în HNP,24
2,reducere LLL disponibilă,False
3,candidat returnat,False
4,candidat validat,False
5,mesaj reducere,Mai puțin de 40 de predicții CNN disponibile.


## Verificarea aritmeticii din companion paper

Proiectul păstrează și verificările numerice pentru constanta de aproximare, transferul exponentului, proprietatea celor trei goluri și corespondența dintre soluțiile diofantice și intersecțiile geometrice.

In [7]:
import contextlib
import io

bridge_results_path = ARTIFACTS_DIR / "bridge_results.json"
RERUN_MATH = False

if RERUN_MATH or not bridge_results_path.exists():
    with contextlib.redirect_stdout(io.StringIO()):
        bridge_1 = bridge.run_module_1()
        bridge_2 = bridge.run_module_2()
        bridge_3 = bridge.run_module_3()
        bridge_4 = bridge.run_module_4()
    bridge_data = {
        "module_1": bridge_1,
        "module_2": bridge_2,
        "module_3": bridge_3,
        "module_4": bridge_4,
    }
    bridge_results_path.write_text(json.dumps(bridge_data, indent=2), encoding="utf-8")
else:
    bridge_data = json.loads(bridge_results_path.read_text(encoding="utf-8"))

bridge_rows = [
    ["Asimptotic constant", bridge_data["module_1"]["passed"], bridge_data["module_1"]["sqrt2_empirical"], bridge_data["module_1"]["sqrt2_theoretical"]],
    ["Exponent transfer", bridge_data["module_2"]["passed"], bridge_data["module_2"]["mu_geo"], 2.0],
    ["Three-gap rigidity", bridge_data["module_3"]["passed"], bridge_data["module_3"]["gap_counts"][-1], "<= 3"],
    ["Concordanță geometrică", bridge_data["module_4"]["passed"], bridge_data["module_4"]["count_geom"], bridge_data["module_4"]["expected_count"]],
]

bridge_df = pd.DataFrame(
    bridge_rows,
    columns=["modul", "validat", "empiric", "referință"],
)

display(bridge_df)
assert bridge_df["validat"].all()


,modul,validat,empiric,referință
0,Asimptotic constant,True,0.251300,0.25
1,Exponent transfer,True,1.998299,2.0
2,Three-gap rigidity,True,3.000000,<= 3
3,Concordanță geometrică,True,321.000000,321


In [8]:
import contextlib
import io

dioph_results_path = ARTIFACTS_DIR / "diophantine_results.json"

if not dioph_results_path.exists():
    with contextlib.redirect_stdout(io.StringIO()):
        dio = {
            "module_1": diophantine.run_module_1(),
            "module_2": diophantine.run_module_2(),
            "module_3": diophantine.run_module_3(),
            "module_4": diophantine.run_module_4(),
            "module_5": diophantine.run_module_5(),
        }
    dioph_results_path.write_text(json.dumps(dio, indent=2), encoding="utf-8")
else:
    dio = json.loads(dioph_results_path.read_text(encoding="utf-8"))

dio_df = pd.DataFrame([
    ["Modul 1", dio["module_1"]["passed"]],
    ["Modul 2", dio["module_2"]["passed"]],
    ["Modul 3", dio["module_3"]["passed"]],
    ["Modul 4", dio["module_4"]["passed"]],
    ["Modul 5", dio["module_5"]["passed"]],
], columns=["experiment", "validat"])

display(dio_df)
assert dio_df["validat"].all()


,experiment,validat
0,Modul 1,True
1,Modul 2,True
2,Modul 3,True
3,Modul 4,True
4,Modul 5,True


## Tabloul final

Raportăm separat ce a fost măsurat și ce a rămas neexecutat. Formula HNP poate fi validată chiar dacă reducerea LLL și recuperarea cheii nu rulează în mediul curent.

In [9]:
final_rows = [
    ["Date", "PASS", 160, "semnături generate și verificate"],
    ["Scurgere", "PASS", 160, "urme HW cu σ = 0,15"],
    ["Baseline", "MEASURED", baseline_results["fixed_profile"]["bit_accuracy"], "acuratețe pe bit, set profilat"],
    ["CNN", "MEASURED", cnn_metrics["test_bit_accuracy"], "acuratețe pe bit, set test"],
    ["CNN prefix", "MEASURED", cnn_metrics["test_prefix_accuracy"], "prefix de 12 biți perfect"],
    ["HNP formulat", "VALIDATED", relation_ok, "vectorul țintă este compatibil cu cheia oracle"],
    ["HNP LLL", "EXECUTED" if reduced_available else "NOT EXECUTED", reduced_available, "depinde de fpylll"],
    ["Recuperare cheie", "PASS" if cnn_candidate_valid else "NOT RECOVERED", cnn_candidate_valid, "doar un candidat LLL valid este acceptat"],
    ["Matematică", "PASS", True, "modulele de verificare numerică"],
]

final_df = pd.DataFrame(
    final_rows,
    columns=["etapă", "status", "valoare", "descriere"],
)

display(final_df)

,etapă,status,valoare,descriere
0,Date,PASS,160,semnături generate și verificate
1,Scurgere,PASS,160,"urme HW cu σ = 0,15"
2,Baseline,MEASURED,0.997917,"acuratețe pe bit, set profilat"
3,CNN,MEASURED,0.982639,"acuratețe pe bit, set test"
4,CNN prefix,MEASURED,0.791667,prefix de 12 biți perfect
5,HNP formulat,VALIDATED,True,vectorul țintă este compatibil cu cheia oracle
6,HNP LLL,NOT EXECUTED,False,depinde de fpylll
7,Recuperare cheie,NOT RECOVERED,False,doar un candidat LLL valid este acceptat
8,Matematică,PASS,True,modulele de verificare numerică


## Comparație cu artefactul final existent în repository

Repository-ul conține deja o tabelă de rezultate pentru 160 de eșantioane și 12 biți. Acea versiune raportează 96,354% acuratețe CNN pe bit și 65,625% prefix exact pe setul de validare folosit de scriptul `train.py`. HNP apare validat la nivel de ecuații, fără recuperarea cheii, deoarece `fpylll` nu era disponibil.

Notebook-ul folosește același split explicit ca scripturile CNN: 96 train, 24 validation și 40 test/attack. Valorile CNN sunt raportate pe aceste seturi, iar cele 40 de predicții pot alimenta etapa HNP.

In [10]:
existing_csv = RESULTS_DIR / "tables" / "final-results.csv"
if existing_csv.exists():
    existing = pd.read_csv(existing_csv)
    display(existing)

,stage,status,metric,value,reference
0,dataset,PASS,samples,160,160
1,leakage,PASS,traces,160,160
2,leakage,PASS,trace_width_bits,256,256
3,cnn,PASS,validation_bit_accuracy,0.9635416865348816,measured
4,cnn,PASS,validation_prefix_accuracy,0.65625,measured
5,hnp,VALIDATED,samples,40,40
6,hnp,VALIDATED,recovered_private_key,NaN,not recovered without fpylll
7,hnp,VALIDATED,validated,False,True
8,math,PASS,bridge_modules_passed,4,4
9,math,PASS,diophantine_modules_passed,5,5


In [11]:
final_payload = {
    "cnn": cnn_metrics,
    "baseline": baseline_results["fixed_profile"],
    "hnp_relation_validated": bool(relation_ok),
    "fpylll_available": bool(FPYLLL_AVAILABLE),
    "hnp_cnn_candidate": cnn_recovery,
    "hnp_cnn_candidate_validated": bool(cnn_candidate_valid),
    "math_bridge_all_pass": bool(bridge_df["validat"].all() and dio_df["validat"].all()),
}

(ARTIFACTS_DIR / "notebook_final_summary.json").write_text(
    json.dumps(final_payload, indent=2),
    encoding="utf-8",
)

final_df.to_csv(
    RESULTS_DIR / "tables" / "notebook-final-results.csv",
    index=False,
)

print("Rezumatul final a fost salvat în informatca/artifacts/notebook_final_summary.json".replace("informatca", "informatica"))
print("Tabelul final a fost salvat în results/tables/notebook-final-results.csv")

Rezumatul final a fost salvat în informatica/artifacts/notebook_final_summary.json
Tabelul final a fost salvat în results/tables/notebook-final-results.csv


## Concluzia experimentală

Lanțul de simulare este verificat de la generarea ECDSA până la construirea instanței HNP. CNN-ul produce prefixe discrete care pot fi transformate în termenii $t_i$, $u_i$ și $a_i$. Recuperarea numerică a cheii rămâne o etapă separată și cere o instanță cu suficiente semnături, o reducere LLL executată efectiv și validarea unui candidat.

Setul de test oferă 40 de predicții, iar configurația HNP folosește aceleași 40 de semnături. `fpylll` rămâne backend-ul de referință pentru CI; fallback-ul SymPy este raportat separat atunci când rulează local.

## Referințe

- J. Breitner și N. Heninger, [Biased Nonce Sense: Lattice Attacks Against Weak ECDSA Signatures in Cryptocurrencies](https://dblp.org/rec/conf/fc/BreitnerH19.html), Financial Cryptography 2019.
- [`fpylll`](https://fpylll.readthedocs.io/en/latest/), tutorial și reducere LLL.
- [RFC 6979](https://www.rfc-editor.org/rfc/rfc6979), ECDSA și generarea nonce-ului.